# Estudo Comparativo de Classificação Acústica Submarina — Dataset IARA
### Notebook 8: Comparativo Multimodal Global — MEL, DEMON, LOFAR e FUSÃO (MEL + DEMON)
#### Modelos Avaliados: Multi-Layer Perceptron (MLP), SVM Nyström ($m=6000$), Random Forest e CNN 2D PyTorch

Este notebook consolida a investigação experimental mais abrangente do projeto IARA, avaliando sistematicamente a interação entre **quatro representações acústicas** e **quatro grandes famílias de aprendizado de máquina** sob o protocolo oficial de **$5\times2$ validação cruzada (10 folds)** com isolamento estrito de `Ship ID` sobre todos os **1.706 áudios de UO** (*Silva et al.*, IEEE Access 2025).

--- 

### Representações Espectrais Avaliadas:
1. **MEL (256 filtros log-triangulares):** Integração de energia contínua de banda larga ($0\text{ a }11\text{ kHz}$), capturando ruído de cavitação e combustão.
2. **DEMON (253 bins de modulação):** Demodulação de envoltória em baixa frequência ($0\text{ a }50\text{ Hz}$), capturando taxa de giro do eixo ($	ext{RPM}$) e frequência de passagem de pás da hélice.
3. **LOFAR (1024 bins de banda estreita linear):** Espectrograma clássico de alta resolução de raias harmônicas de maquinário.
4. **FUSÃO MULTIMODAL (509 atributos):** Concatenação direta $\text{MEL} (256) + \text{DEMON} (253)$ na mesma grade temporal (janelas de 5,12s sincronizadas).

--- 

### Famílias de Aprendizado de Máquina:
1. **Multi-Layer Perceptron (MLP PyTorch):** Camadas densas com `BatchNorm1d` e `Dropout`.
2. **SVM Nyström ($m=6000$):** Mapeamento em espaço de Hilbert de alta dimensão via kernel RBF Gaussiano com penalização ElasticNet.
3. **Random Forest (100 árvores):** Ensembles baseados em árvores de decisão ortogonais com balanceamento de pesos.
4. **CNN 2D PyTorch (GPU RTX 5060 Ti):** Blocos sequenciais de tensores temporais ($32 \times D$) com convoluções bidimensionais.


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

# Configuração visual acadêmica
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'figure.titlesize': 15,
    'figure.dpi': 130
})

print('Ambiente carregado com sucesso!')


## 1. Tabela Mestra Comparativa Global ($4 \times 4$)

Abaixo apresentamos a matriz completa de resultados dos 16 pares (Modelo $\times$ Representação) para **4 Classes** e **3 Classes**:

In [ ]:
dados_completos = [
    # MLP
    {'Modelo': 'MLP (PyTorch)', 'Representacao': 'DEMON', 'Dim': '253-D', 'Acc 4C (%)': 53.10, 'SP 4C (%)': 53.51, 'Rec SMALL 4C (%)': 34.78, 'Acc 3C (%)': 57.25, 'SP 3C (%)': 55.62, 'Rec SMALL 3C (%)': 46.10, 'Tipo': 'Isolado'},
    {'Modelo': 'MLP (PyTorch)', 'Representacao': 'MEL', 'Dim': '256-D', 'Acc 4C (%)': 64.26, 'SP 4C (%)': 64.93, 'Rec SMALL 4C (%)': 45.62, 'Acc 3C (%)': 64.53, 'SP 3C (%)': 63.98, 'Rec SMALL 3C (%)': 55.11, 'Tipo': 'Isolado'},
    {'Modelo': 'MLP (PyTorch)', 'Representacao': 'LOFAR', 'Dim': '1024-D', 'Acc 4C (%)': 67.48, 'SP 4C (%)': 66.51, 'Rec SMALL 4C (%)': 47.80, 'Acc 3C (%)': 65.50, 'SP 3C (%)': 64.20, 'Rec SMALL 3C (%)': 52.30, 'Tipo': 'Baseline Artigo'},
    {'Modelo': 'MLP (PyTorch)', 'Representacao': 'FUSÃO (MEL+DEMON)', 'Dim': '509-D', 'Acc 4C (%)': 66.18, 'SP 4C (%)': 66.99, 'Rec SMALL 4C (%)': 53.93, 'Acc 3C (%)': 66.93, 'SP 3C (%)': 66.18, 'Rec SMALL 3C (%)': 60.90, 'Tipo': 'Proposto (Fusão)'},
    
    # SVM m=6000
    {'Modelo': 'SVM Nyström (m=6000)', 'Representacao': 'DEMON', 'Dim': '253-D', 'Acc 4C (%)': 51.72, 'SP 4C (%)': 46.57, 'Rec SMALL 4C (%)': 15.51, 'Acc 3C (%)': 57.80, 'SP 3C (%)': 53.25, 'Rec SMALL 3C (%)': 57.92, 'Tipo': 'Isolado'},
    {'Modelo': 'SVM Nyström (m=6000)', 'Representacao': 'MEL', 'Dim': '256-D', 'Acc 4C (%)': 64.56, 'SP 4C (%)': 63.78, 'Rec SMALL 4C (%)': 45.60, 'Acc 3C (%)': 64.02, 'SP 3C (%)': 63.50, 'Rec SMALL 3C (%)': 55.00, 'Tipo': 'Isolado'},
    {'Modelo': 'SVM Nyström (m=6000)', 'Representacao': 'LOFAR', 'Dim': '1024-D', 'Acc 4C (%)': 65.47, 'SP 4C (%)': 63.83, 'Rec SMALL 4C (%)': 42.58, 'Acc 3C (%)': 59.41, 'SP 3C (%)': 60.59, 'Rec SMALL 3C (%)': 47.81, 'Tipo': 'Proposto (SVM LOFAR)'},
    {'Modelo': 'SVM Nyström (m=6000)', 'Representacao': 'FUSÃO (MEL+DEMON)', 'Dim': '509-D', 'Acc 4C (%)': 64.55, 'SP 4C (%)': 64.85, 'Rec SMALL 4C (%)': 47.08, 'Acc 3C (%)': 64.58, 'SP 3C (%)': 63.33, 'Rec SMALL 3C (%)': 60.90, 'Tipo': 'Proposto (Fusão)'},
    
    # Random Forest
    {'Modelo': 'Random Forest (100 arv.)', 'Representacao': 'DEMON', 'Dim': '253-D', 'Acc 4C (%)': 50.06, 'SP 4C (%)': 44.30, 'Rec SMALL 4C (%)': 53.40, 'Acc 3C (%)': 56.40, 'SP 3C (%)': 52.10, 'Rec SMALL 3C (%)': 60.51, 'Tipo': 'Isolado'},
    {'Modelo': 'Random Forest (100 arv.)', 'Representacao': 'MEL', 'Dim': '256-D', 'Acc 4C (%)': 62.64, 'SP 4C (%)': 62.22, 'Rec SMALL 4C (%)': 44.20, 'Acc 3C (%)': 62.10, 'SP 3C (%)': 61.50, 'Rec SMALL 3C (%)': 52.40, 'Tipo': 'Baseline Artigo'},
    {'Modelo': 'Random Forest (100 arv.)', 'Representacao': 'LOFAR', 'Dim': '1024-D', 'Acc 4C (%)': 58.87, 'SP 4C (%)': 56.92, 'Rec SMALL 4C (%)': 39.10, 'Acc 3C (%)': 57.50, 'SP 3C (%)': 55.20, 'Rec SMALL 3C (%)': 45.60, 'Tipo': 'Baseline Artigo'},
    {'Modelo': 'Random Forest (100 arv.)', 'Representacao': 'FUSÃO (MEL+DEMON)', 'Dim': '509-D', 'Acc 4C (%)': 61.99, 'SP 4C (%)': 62.05, 'Rec SMALL 4C (%)': 59.44, 'Acc 3C (%)': 61.77, 'SP 3C (%)': 61.16, 'Rec SMALL 3C (%)': 59.83, 'Tipo': 'Proposto (Fusão)'},
    
    # CNN 2D
    {'Modelo': 'CNN 2D PyTorch', 'Representacao': 'DEMON', 'Dim': '32x253', 'Acc 4C (%)': 42.47, 'SP 4C (%)': 35.72, 'Rec SMALL 4C (%)': 19.33, 'Acc 3C (%)': 47.10, 'SP 3C (%)': 32.15, 'Rec SMALL 3C (%)': 22.40, 'Tipo': 'Isolado'},
    {'Modelo': 'CNN 2D PyTorch', 'Representacao': 'MEL', 'Dim': '32x256', 'Acc 4C (%)': 64.99, 'SP 4C (%)': 63.52, 'Rec SMALL 4C (%)': 46.20, 'Acc 3C (%)': 65.10, 'SP 3C (%)': 63.80, 'Rec SMALL 3C (%)': 54.80, 'Tipo': 'Baseline Artigo'},
    {'Modelo': 'CNN 2D PyTorch', 'Representacao': 'LOFAR', 'Dim': '32x1024', 'Acc 4C (%)': 67.02, 'SP 4C (%)': 66.05, 'Rec SMALL 4C (%)': 47.10, 'Acc 3C (%)': 65.80, 'SP 3C (%)': 64.50, 'Rec SMALL 3C (%)': 53.10, 'Tipo': 'Baseline Artigo'},
    {'Modelo': 'CNN 2D PyTorch', 'Representacao': 'FUSÃO (MEL+DEMON)', 'Dim': '32x509', 'Acc 4C (%)': 38.22, 'SP 4C (%)': 23.96, 'Rec SMALL 4C (%)': 28.09, 'Acc 3C (%)': 43.92, 'SP 3C (%)': 22.03, 'Rec SMALL 3C (%)': 33.15, 'Tipo': 'Proposto (Fusão)'}
]

df_master = pd.DataFrame(dados_completos)
display(df_master)


## 2. Gráfico 1: O Impacto da Fusão MEL + DEMON vs. LOFAR (Índice SP Oficial)

Abaixo plotamos a comparação de todas as representações para os 4 modelos:

In [ ]:
fig, ax = plt.subplots(figsize=(15, 7))

models = ['MLP (PyTorch)', 'SVM Nyström (m=6000)', 'Random Forest (100 arv.)', 'CNN 2D PyTorch']
x = np.arange(len(models))
width = 0.20

# Coletar SP por representação
sp_demon = [df_master[(df_master['Modelo'] == m) & (df_master['Representacao'] == 'DEMON')]['SP 4C (%)'].values[0] for m in models]
sp_mel   = [df_master[(df_master['Modelo'] == m) & (df_master['Representacao'] == 'MEL')]['SP 4C (%)'].values[0] for m in models]
sp_lofar = [df_master[(df_master['Modelo'] == m) & (df_master['Representacao'] == 'LOFAR')]['SP 4C (%)'].values[0] for m in models]
sp_fused = [df_master[(df_master['Modelo'] == m) & (df_master['Representacao'] == 'FUSÃO (MEL+DEMON)')]['SP 4C (%)'].values[0] for m in models]

r1 = ax.bar(x - 1.5*width, sp_demon, width, label='DEMON Puro (253-D)', color='#95a5a6', alpha=0.9)
r2 = ax.bar(x - 0.5*width, sp_mel,   width, label='MEL Puro (256-D)',   color='#3498db', alpha=0.9)
r3 = ax.bar(x + 0.5*width, sp_lofar, width, label='LOFAR Puro (1024-D)', color='#e67e22', alpha=0.9)
r4 = ax.bar(x + 1.5*width, sp_fused, width, label='FUSÃO MEL+DEMON (509-D)', color='#27ae60', alpha=0.95)

ax.set_title('Comparativo Global do Índice SP Oficial (%): 4 Modelos × 4 Representações (5x2cv)', fontweight='bold', fontsize=14, pad=15)
ax.set_ylabel('Índice SP Oficial (%)', fontweight='bold', fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels(models, fontweight='bold', fontsize=11)
ax.set_ylim(0, 78)
ax.legend(loc='lower left', frameon=True, fontsize=10.5)

for rects in [r1, r2, r3, r4]:
    for rect in rects:
        h = rect.get_height()
        c = '#c0392b' if h < 30 else 'black'
        weight = 'bold' if rects == r4 else 'normal'
        ax.annotate(f'{h:.1f}%', xy=(rect.get_x() + rect.get_width()/2, h),
                    xytext=(0, 4), textcoords="offset points", ha='center', va='bottom', fontsize=9, fontweight=weight, color=c)

plt.tight_layout()
plt.show()


## 3. Gráfico 2: Desbloqueio da Classe Crítica `SMALL` (Lanchas Rápidas)

O grande desafio histórico do dataset IARA sempre foi identificar embarcações pequenas (`SMALL`), cuja sensibilidade no LOFAR e no MEL nunca passava de $45\%\text{--}47\%$. Veja abaixo o impacto do DEMON na sensibilidade de `SMALL`:

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# 4 Classes
sm_4c_mel   = [df_master[(df_master['Modelo'] == m) & (df_master['Representacao'] == 'MEL')]['Rec SMALL 4C (%)'].values[0] for m in models[:3]]
sm_4c_lofar = [df_master[(df_master['Modelo'] == m) & (df_master['Representacao'] == 'LOFAR')]['Rec SMALL 4C (%)'].values[0] for m in models[:3]]
sm_4c_fused = [df_master[(df_master['Modelo'] == m) & (df_master['Representacao'] == 'FUSÃO (MEL+DEMON)')]['Rec SMALL 4C (%)'].values[0] for m in models[:3]]

x_vec = np.arange(len(models[:3]))
w = 0.25

ax1.bar(x_vec - w, sm_4c_mel, w, label='MEL Puro', color='#3498db')
ax1.bar(x_vec, sm_4c_lofar, w, label='LOFAR Puro', color='#e67e22')
r_f = ax1.bar(x_vec + w, sm_4c_fused, w, label='FUSÃO MEL+DEMON', color='#27ae60')
ax1.set_title('Sensibilidade em SMALL — 4 Classes (%)', fontweight='bold')
ax1.set_xticks(x_vec)
ax1.set_xticklabels(models[:3], fontweight='bold')
ax1.set_ylim(0, 75)
ax1.legend()

for rect in r_f:
    h = rect.get_height()
    ax1.annotate(f'{h:.1f}%', xy=(rect.get_x() + rect.get_width()/2, h),
                 xytext=(0, 4), textcoords="offset points", ha='center', va='bottom', fontsize=9.5, fontweight='bold', color='#27ae60')

# 3 Classes (Apenas Embarcações)
sm_3c_mel   = [df_master[(df_master['Modelo'] == m) & (df_master['Representacao'] == 'MEL')]['Rec SMALL 3C (%)'].values[0] for m in models[:3]]
sm_3c_lofar = [df_master[(df_master['Modelo'] == m) & (df_master['Representacao'] == 'LOFAR')]['Rec SMALL 3C (%)'].values[0] for m in models[:3]]
sm_3c_fused = [df_master[(df_master['Modelo'] == m) & (df_master['Representacao'] == 'FUSÃO (MEL+DEMON)')]['Rec SMALL 3C (%)'].values[0] for m in models[:3]]

ax2.bar(x_vec - w, sm_3c_mel, w, label='MEL Puro', color='#3498db')
ax2.bar(x_vec, sm_3c_lofar, w, label='LOFAR Puro', color='#e67e22')
r_f3 = ax2.bar(x_vec + w, sm_3c_fused, w, label='FUSÃO MEL+DEMON', color='#27ae60')
ax2.set_title('Sensibilidade em SMALL — 3 Classes (Embarcações) (%)', fontweight='bold')
ax2.set_xticks(x_vec)
ax2.set_xticklabels(models[:3], fontweight='bold')
ax2.set_ylim(0, 75)
ax2.legend()

for rect in r_f3:
    h = rect.get_height()
    ax2.annotate(f'{h:.1f}%', xy=(rect.get_x() + rect.get_width()/2, h),
                 xytext=(0, 4), textcoords="offset points", ha='center', va='bottom', fontsize=9.5, fontweight='bold', color='#27ae60')

plt.tight_layout()
plt.show()


## 4. Matrizes de Confusão Normalizadas da Fusão (MLP e SVM)

Abaixo comparamos as matrizes de confusão do campeão geral (**MLP na Fusão**) e do **SVM ($m=6000$)**:

In [ ]:
cm_mlp_4c = np.array([
    [53.9, 23.0, 17.4,  5.7],
    [23.8, 63.6, 10.3,  2.3],
    [14.9, 11.0, 71.5,  2.7],
    [11.2,  4.9,  3.1, 80.8]
])

cm_svm_4c = np.array([
    [47.1, 25.3, 20.5,  7.1],
    [21.7, 65.1, 10.7,  2.6],
    [13.6, 11.3, 72.1,  3.0],
    [ 9.7,  5.6,  5.7, 79.0]
])

classes = ['SMALL', 'MEDIUM', 'LARGE', 'AMBIENTAL']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

im1 = ax1.imshow(cm_mlp_4c, cmap='Blues', vmin=0, vmax=100)
ax1.set_title('Matriz de Confusão: MLP na FUSÃO MEL+DEMON (SP = 66.99%)', fontweight='bold')
ax1.set_xticks(range(4)); ax1.set_yticks(range(4))
ax1.set_xticklabels(classes, fontweight='bold'); ax1.set_yticklabels(classes, fontweight='bold')
for i in range(4):
    for j in range(4):
        val = cm_mlp_4c[i, j]
        ax1.text(j, i, f'{val:.1f}%', ha='center', va='center', color='white' if val > 45 else 'black', fontweight='bold' if i == j else 'normal')

im2 = ax2.imshow(cm_svm_4c, cmap='Greens', vmin=0, vmax=100)
ax2.set_title('Matriz de Confusão: SVM (m=6000) na FUSÃO MEL+DEMON (SP = 64.85%)', fontweight='bold')
ax2.set_xticks(range(4)); ax2.set_yticks(range(4))
ax2.set_xticklabels(classes, fontweight='bold'); ax2.set_yticklabels(classes, fontweight='bold')
for i in range(4):
    for j in range(4):
        val = cm_svm_4c[i, j]
        ax2.text(j, i, f'{val:.1f}%', ha='center', va='center', color='white' if val > 45 else 'black', fontweight='bold' if i == j else 'normal')

plt.tight_layout()
plt.show()


## 5. Conclusões Físicas e Arquiteturais Fundamentais

1. **Por que a Fusão MEL + DEMON equivale ou supera o LOFAR?**
   * O **LOFAR (1024-D)** analisa a banda audível estreita. Ele retém tanto ruído contínuo quanto linhas discretas de maquinário.
   * A **Fusão MEL + DEMON (509-D)** decompõe o problema acusticamente: o **MEL** fornece a energia contínua de cavitação/combustão e o **DEMON** fornece a frequência de giro do eixo e número de pás.
   * O resultado é a mesma capacidade discriminativa do LOFAR, mas com **metade da dimensão** ($509$ vs. $1024$) e muito maior equilíbrio para alvos rápidos (`SMALL`).

2. **Por que a CNN 2D colapsa na Fusão direta?**
   * Redes convolucionais assumem **correlação local contínua** (pixels vizinhos têm continuidade semântica).
   * Ao concatenar os 256 canais de Mel com os 253 canais de DEMON, a fronteira artificial entre eles não tem coerência física. Convoluções 2D agem como filtros passa-baixas que borram os harmônicos finos do DEMON e poluem o Mel, provocando colapso de gradiente em vários folds ($SP = 23{,}96\%$).

3. **Modelos Vetoriais (MLP e SVM) são os Vencedores Naturais da Fusão Multimodal:**
   * Tratando os dados como vetores em $\mathbb{R}^{509}$, o **MLP PyTorch** estabelece o recorde absoluto de **$66{,}99\% \pm 1{,}67\%$ de Índice SP**, superando a MLP LOFAR ($66{,}51\%$) e a CNN LOFAR ($66{,}05\%$) do artigo original.
   * O **SVM Nyström ($m=6000$)** sobe para **$64{,}85\%$ de SP**, superando seu benchmark com LOFAR ($63{,}83\%$) e atingindo **$60{,}90\%$ de recall em SMALL** no problema de embarcações.